# The REG102_PAT_PUBLN table

Welcome to the second table of PATSTAT Register, namely table **REG102_PAT_PUBLN**. This table contains the EP and WO publications of the EP applications and international applications of table REG101_APPLN.

Here we can easily keep track of data concerning patent publication, such as publication date, publication kind, and publication in the European Patent Bulletin.

In [1]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG102_PAT_PUBLN
from sqlalchemy import func
import pandas as pd

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## ID (Primary Key)

Technical identifier for an application, without business meaning. Its values will not change from one PATSTAT edition to the next.

In [2]:
i = db.query(
    REG102_PAT_PUBLN.id
).limit(1000)

df = patstat.df(i)
df

,id
0,14877362
1,971925
2,19868002
3,14741704
4,21909274
...,...
995,17913648
996,5808742
997,19736121
998,80107343


## BULLETIN_YEAR

For actions that have been published in the European Patent Bulletin, it is the year of the publication in the European Patent Bulletin. The default value is 0, used for applications that are not published or for which the year is not known. The format is YYYY otherwise.

In [3]:
years = db.query(
    REG102_PAT_PUBLN.bulletin_year,
    REG102_PAT_PUBLN.id
).limit(1000)

years_df = patstat.df(years)
years_df

,bulletin_year,id
0,2018,14877362
1,2002,971925
2,2020,19868002
3,2015,14741704
4,2022,21909274
...,...,...
995,2021,17913648
996,2006,5808742
997,2019,19736121
998,1981,80107343


## BULLETIN_NR

This is the issue number of the European Patent Bulletin for actions that have been published in it. This attribute indicates the calendar week the European Patent Bulletin was published. The default value 0 is used when the attribute `bulletin_year` is 0.

In [4]:
bulletin_nr = db.query(
    REG102_PAT_PUBLN.id,
    REG102_PAT_PUBLN.bulletin_nr,
    REG102_PAT_PUBLN.bulletin_year
).limit(100)

bulletin_nr_df = patstat.df(bulletin_nr)
bulletin_nr_df

,id,bulletin_nr,bulletin_year
0,14877362,34,2018
1,971925,37,2002
2,19868002,14,2020
3,14741704,2,2015
4,21909274,26,2022
...,...,...,...
95,22823818,51,2022
96,2009306,50,2003
97,97203588,20,1999
98,14713436,5,2016


We can see that there are more than six thousand applications with `bulletin_nr` equal to 0, i.e. applications with `bulletin_year` equal to 0.

In [5]:
default_value = db.query(
    REG102_PAT_PUBLN.id,
    REG102_PAT_PUBLN.bulletin_nr,
    REG102_PAT_PUBLN.bulletin_year
).filter(
    REG102_PAT_PUBLN.bulletin_year == 0
)

default_value_df = patstat.df(default_value)
default_value_df

,id,bulletin_nr,bulletin_year
0,78200044,0,0
1,79100888,0,0
2,79100783,0,0
3,78300480,0,0
4,79101477,0,0
...,...,...,...
6395,78101106,0,0
6396,78300134,0,0
6397,79100534,0,0
6398,78400239,0,0


## PUBLN_AUTH

Publication authority, which is either the EPO or WIPO.

In [6]:
auth = db.query(
    REG102_PAT_PUBLN.publn_auth,
    func.count(REG102_PAT_PUBLN.id).label('number_of_applications')
).group_by(
    REG102_PAT_PUBLN.publn_auth
).order_by(
    func.count(REG102_PAT_PUBLN.id).label('number_of_applications').desc()
)

auth_df = patstat.df(auth)
auth_df

,publn_auth,number_of_applications
0,EP,7836176
1,WO,5154176


## PUBLN_NR

Document number for publication. This number consists of up to ten digits. Leading zeros are significant, so this attribute must be used as a text string not as a numerical value. EP publications numbers always consist of seven digits.

In [7]:
publn_nr = db.query(
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_auth
).limit(1000)

publn_nr_df = patstat.df(publn_nr)
publn_nr_df

,publn_nr,publn_auth
0,2023286960,WO
1,2013132677,WO
2,2861354,EP
3,0242690,EP
4,2204698,EP
...,...,...
995,0972138,EP
996,9945452,WO
997,3190138,EP
998,2009114458,WO


## PUBLN_KIND

Publication kind code: up to two characters consisting of A or B followed by a digit.

Let's count the number of occurrences of each combination `publn_kind`–`publn_auth`.

In [8]:
kind = db.query(
    REG102_PAT_PUBLN.publn_kind,
    REG102_PAT_PUBLN.publn_auth,
    func.count(REG102_PAT_PUBLN.id).label('number_of_applications')
).group_by(
    REG102_PAT_PUBLN.publn_kind,
    REG102_PAT_PUBLN.publn_auth
).order_by(
    func.count(REG102_PAT_PUBLN.id)
)

kind_df = patstat.df(kind)
kind_df

,publn_kind,publn_auth,number_of_applications
0,B3,EP,602
1,A9,EP,2865
2,A8,EP,4672
3,B9,EP,7489
4,B8,EP,21613
5,B2,EP,32061
6,A2,WO,656044
7,A3,EP,691136
8,A2,EP,1066135
9,B1,EP,2467634


## PUBLN_DATE

Date of publication. As with other attributes, there may be missing dates.

In [9]:
publn_date = db.query(
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date
).limit(1000)

publn_date_df = patstat.df(publn_date)
publn_date_df

,publn_nr,publn_date
0,2023286960,2023-01-19
1,2013132677,2013-09-12
2,2861354,2015-04-22
3,0242690,1989-01-18
4,2204698,2018-08-08
...,...,...
995,0972138,2003-05-02
996,9945452,1999-09-10
997,3190138,2019-10-16
998,2009114458,2009-09-17


It is possible to retrieve the total number of publications that occurred within a specific time window, for example after 2022.

In [10]:
recent_publn = db.query(
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date
).filter(
    REG102_PAT_PUBLN.publn_date > '2022-12-31'
)

recent_publn_df = patstat.df(recent_publn)
recent_publn_df

,publn_nr,publn_date
0,4301117,2024-01-03
1,2024056274,2024-03-21
2,3580910,2024-07-10
3,3647758,2024-05-08
4,4560317,2025-05-28
...,...,...
1545831,4597587,2025-08-06
1545832,4151613,2025-08-06
1545833,4597516,2025-08-06
1545834,4593978,2025-08-06


## PUBLN_LG

Language of publication. The domain consists of up to two characters, according to ISO 639-1 language codes.

We can rank the most frequent languages.

In [11]:
lang = db.query(
    REG102_PAT_PUBLN.publn_lg,
    func.count(REG102_PAT_PUBLN.id).label('number_of_applications')
).group_by(
    REG102_PAT_PUBLN.publn_lg
).order_by(
    func.count(REG102_PAT_PUBLN.id).desc()
).limit(15)

lang_df = patstat.df(lang)
lang_df

,publn_lg,number_of_applications
0,en,8447987
1,de,1875082
2,ja,788984
3,zh,602674
4,fr,599994
5,,396685
6,ko,212463
7,es,27495
8,ru,21031
9,pt,7443


We can combine the language search with the date. Let's say that we are interested in the number of publications in English in 2023.

In [12]:
year_lang = db.query(
    REG102_PAT_PUBLN.publn_nr,
    REG102_PAT_PUBLN.publn_date
).filter(
    REG102_PAT_PUBLN.publn_date > '2022-12-31',
    REG102_PAT_PUBLN.publn_date < '2024-01-01',
    REG102_PAT_PUBLN.publn_lg == 'en'
)

year_lang_df = patstat.df(year_lang)
year_lang_df

,publn_nr,publn_date
0,2023018809,2023-02-16
1,2023018895,2023-02-16
2,2023150563,2023-08-10
3,2023137244,2023-07-20
4,4125021,2023-02-01
...,...,...
367075,2023249809,2023-12-28
367076,2023247701,2023-12-28
367077,2023247031,2023-12-28
367078,2023249990,2023-12-28
